<a href="https://colab.research.google.com/github/fmaignacio/datajud_probe/blob/claude%2Fevaluate-legal-databases-yWU5P/09_datajud_stj_por_cnj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 09 - DataJud STJ por CNJ

Objetivo: consultar o endpoint `api_publica_stj` do DataJud usando os CNJs presentes em `stj_processos_ciclo_vida.parquet` e salvar uma camada estruturada de metadados e movimentos.

Este notebook deve trabalhar com lote pequeno primeiro e so depois escalar.

## 1. Ambiente e parametros

In [ ]:
!find /content/drive/MyDrive/Mestrado/2026/llms/data/raw/datajud_stj_lookup -maxdepth 1 -type f -name "*.json" | wc -l

2582


In [ ]:
raw_rows = []

for _, row in tqdm(processos_consulta.iterrows(), total=len(processos_consulta), desc='rebuild lookup_status from raw'):
    numero_processo = str(row['numero_processo'])
    numero_registro_stj = row.get('numero_registro_stj')
    raw_path = RAW_DATAJUD_STJ / f'{numero_processo}.json'

    status = 'ok'
    error = None
    data = None

    if raw_path.exists() and raw_path.stat().st_size > 0:
        try:
            data = json.loads(raw_path.read_text(encoding='utf-8'))
        except Exception as exc:
            status = 'erro'
            error = f'erro_lendo_json_local: {exc}'
    else:
        status = 'erro'
        error = 'json_nao_encontrado_no_raw'

    src = first_hit_source(data) if data else None
    raw_rows.append({
        'numero_processo': numero_processo,
        'numero_registro_stj': numero_registro_stj,
        'status_datajud': status,  # manter 'ok' para funcionar no restante do notebook
        'n_hits': len(data.get('hits', {}).get('hits', [])) if data else 0,
        'data_ajuizamento_datajud': src.get('dataAjuizamento') if src else None,
        'data_ultima_atualizacao_datajud': src.get('dataHoraUltimaAtualizacao') if src else None,
        'classe_nome_datajud': (src.get('classe') or {}).get('nome') if src else None,
        'classe_codigo_datajud': (src.get('classe') or {}).get('codigo') if src else None,
        'erro_datajud': error,
        'raw_path': str(raw_path) if status == 'ok' else None,
    })

lookup_status = pd.DataFrame(raw_rows)
lookup_status.head()
print(lookup_status['status_datajud'].value_counts(dropna=False))

rebuild lookup_status from raw:   0%|          | 0/2582 [00:00<?, ?it/s]

status_datajud
ok    2582
Name: count, dtype: int64


In [ ]:
import os
os.environ["DATAJUD_API_KEY_PUBLICA"] = "cDZHYzlZa0JadVREZDJCendQbXY6SkJlTzNjLV9TRENyQk1RdnFKZGRQdw=="


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from tqdm.auto import tqdm

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    MOUNT_POINT = Path('/content/drive')
    if not (MOUNT_POINT / 'MyDrive').exists():
        drive.mount(str(MOUNT_POINT), force_remount=True)
    DATA_ROOT = MOUNT_POINT / 'MyDrive/Mestrado/2026/llms/data'
else:
    DATA_ROOT = Path.cwd().resolve().parents[0] / 'data'

PROCESSED_DATA = DATA_ROOT / 'processed'
RAW_DATA = DATA_ROOT / 'raw'
RAW_DATAJUD_STJ = RAW_DATA / 'datajud_stj_lookup'
REPORTS_DATA = DATA_ROOT / 'reports' / 'summaries'

for path in [PROCESSED_DATA, RAW_DATAJUD_STJ, REPORTS_DATA]:
    path.mkdir(parents=True, exist_ok=True)

API_URL = 'https://api-publica.datajud.cnj.jus.br/api_publica_stj/_search'
API_KEY = os.getenv('DATAJUD_API_KEY_PUBLICA', '').strip()
PROCESS_SPINE_PATH = PROCESSED_DATA / 'stj_processos_ciclo_vida.parquet'

MAX_PROCESSOS = None
SLEEP_SECONDS = 0.0

print('DATA_ROOT:', DATA_ROOT)
print('PROCESS_SPINE_PATH:', PROCESS_SPINE_PATH, PROCESS_SPINE_PATH.exists())
print('RAW_DATAJUD_STJ:', RAW_DATAJUD_STJ)
print('API key presente:', bool(API_KEY))

Mounted at /content/drive
DATA_ROOT: /content/drive/MyDrive/Mestrado/2026/llms/data
PROCESS_SPINE_PATH: /content/drive/MyDrive/Mestrado/2026/llms/data/processed/stj_processos_ciclo_vida.parquet True
RAW_DATAJUD_STJ: /content/drive/MyDrive/Mestrado/2026/llms/data/raw/datajud_stj_lookup
API key presente: True


## 2. Carregar processos-base

In [ ]:
process_spine = pd.read_parquet(PROCESS_SPINE_PATH)
process_spine = process_spine.dropna(subset=['numero_processo']).copy()
process_spine['numero_processo'] = process_spine['numero_processo'].astype('string')
processos_consulta = process_spine[['numero_processo', 'numero_registro_stj']].drop_duplicates().copy()
if MAX_PROCESSOS is not None:
    processos_consulta = processos_consulta.head(MAX_PROCESSOS)

print('Processos para consulta:', len(processos_consulta))
processos_consulta.head()

Processos para consulta: 2582


,numero_processo,numero_registro_stj
0,00000094820228272722,202301327446
1,00000195520128020001,202302129700
2,00000270520134047105,201700539640
3,00000461120218120012,202302183487
4,00000675420208190035,202300281603


## 3. Funcoes de consulta

In [ ]:
def fetch_datajud_stj(numero_processo: str) -> dict[str, Any]:
    headers = {
        'Authorization': f'ApiKey {API_KEY}',
        'Content-Type': 'application/json',
    }
    payload = {
        'query': {
            'match': {
                'numeroProcesso': numero_processo
            }
        },
        'size': 10,
    }
    response = requests.post(API_URL, headers=headers, json=payload, timeout=120)
    response.raise_for_status()
    return response.json()


def first_hit_source(data: dict[str, Any]) -> dict[str, Any] | None:
    hits = data.get('hits', {}).get('hits', [])
    if not hits:
        return None
    return hits[0].get('_source')


## 4. Coleta bruta por processo

In [ ]:
raw_rows = []

for _, row in tqdm(processos_consulta.iterrows(), total=len(processos_consulta), desc='datajud stj'):
    numero_processo = str(row['numero_processo'])
    numero_registro_stj = row.get('numero_registro_stj')
    raw_path = RAW_DATAJUD_STJ / f'{numero_processo}.json'
    status = 'ok'
    error = None
    data = None
    try:
        data = fetch_datajud_stj(numero_processo)
        raw_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
    except Exception as exc:
        status = 'erro'
        error = str(exc)

    src = first_hit_source(data) if data else None
    raw_rows.append({
        'numero_processo': numero_processo,
        'numero_registro_stj': numero_registro_stj,
        'status_datajud': status,
        'n_hits': len(data.get('hits', {}).get('hits', [])) if data else 0,
        'data_ajuizamento_datajud': src.get('dataAjuizamento') if src else None,
        'data_ultima_atualizacao_datajud': src.get('dataHoraUltimaAtualizacao') if src else None,
        'classe_nome_datajud': (src.get('classe') or {}).get('nome') if src else None,
        'classe_codigo_datajud': (src.get('classe') or {}).get('codigo') if src else None,
        'erro_datajud': error,
        'raw_path': str(raw_path) if status == 'ok' else None,
    })

lookup_status = pd.DataFrame(raw_rows)
lookup_status.head()

datajud stj:   0%|          | 0/2582 [00:00<?, ?it/s]

: 

## 5. Extrair processos, movimentos e assuntos

In [ ]:
process_rows = []
mov_rows = []
assunto_rows = []

for _, row in lookup_status[lookup_status['status_datajud'].eq('ok')].iterrows():
    data = json.loads(Path(row['raw_path']).read_text(encoding='utf-8'))
    src = first_hit_source(data)
    if not src:
        continue

    process_rows.append({
        'numero_processo': row['numero_processo'],
        'numero_registro_stj': row['numero_registro_stj'],
        'tribunal_datajud': src.get('tribunal'),
        'grau_datajud': src.get('grau'),
        'data_ajuizamento_datajud': src.get('dataAjuizamento'),
        'data_ultima_atualizacao_datajud': src.get('dataHoraUltimaAtualizacao'),
        'classe_codigo_datajud': (src.get('classe') or {}).get('codigo'),
        'classe_nome_datajud': (src.get('classe') or {}).get('nome'),
        'orgao_julgador_codigo_datajud': (src.get('orgaoJulgador') or {}).get('codigo'),
        'orgao_julgador_nome_datajud': (src.get('orgaoJulgador') or {}).get('nome'),
        'nivel_sigilo_datajud': src.get('nivelSigilo'),
    })

    for idx, mov in enumerate(src.get('movimentos') or []):
        mov_rows.append({
            'numero_processo': row['numero_processo'],
            'numero_registro_stj': row['numero_registro_stj'],
            'movimento_idx': idx,
            'codigo_movimento': mov.get('codigo'),
            'nome_movimento': mov.get('nome'),
            'data_movimento': mov.get('dataHora'),
            'complementos_tabelados': json.dumps(mov.get('complementosTabelados'), ensure_ascii=False) if mov.get('complementosTabelados') is not None else None,
            'orgao_julgador_codigo_movimento': (mov.get('orgaoJulgador') or {}).get('codigo'),
            'orgao_julgador_nome_movimento': (mov.get('orgaoJulgador') or {}).get('nome'),
        })

    for assunto in src.get('assuntos') or []:
        assunto_rows.append({
            'numero_processo': row['numero_processo'],
            'numero_registro_stj': row['numero_registro_stj'],
            'assunto_codigo_datajud': assunto.get('codigo') if isinstance(assunto, dict) else None,
            'assunto_nome_datajud': assunto.get('nome') if isinstance(assunto, dict) else None,
        })

stj_datajud_processos = pd.DataFrame(process_rows)
stj_datajud_movimentos = pd.DataFrame(mov_rows)
stj_datajud_assuntos = pd.DataFrame(assunto_rows)

print('processos:', stj_datajud_processos.shape)
print('movimentos:', stj_datajud_movimentos.shape)
print('assuntos:', stj_datajud_assuntos.shape)


processos: (2393, 11)
movimentos: (65819, 9)
assuntos: (4081, 4)


## 6. Salvar artefatos

In [ ]:
lookup_status.to_parquet(PROCESSED_DATA / 'stj_datajud_lookup_status.parquet', index=False)
if not stj_datajud_processos.empty:
    stj_datajud_processos.to_parquet(PROCESSED_DATA / 'stj_datajud_processos.parquet', index=False)
if not stj_datajud_movimentos.empty:
    stj_datajud_movimentos.to_parquet(PROCESSED_DATA / 'stj_datajud_movimentos.parquet', index=False)
if not stj_datajud_assuntos.empty:
    stj_datajud_assuntos.to_parquet(PROCESSED_DATA / 'stj_datajud_assuntos.parquet', index=False)

print('Salvo em:', PROCESSED_DATA)

Salvo em: /content/drive/MyDrive/Mestrado/2026/llms/data/processed


In [ ]:
stj_datajud_processos.dtypes


,0
numero_processo,object
numero_registro_stj,object
tribunal_datajud,object
grau_datajud,object
data_ajuizamento_datajud,object
data_ultima_atualizacao_datajud,object
classe_codigo_datajud,int64
classe_nome_datajud,object
orgao_julgador_codigo_datajud,int64
orgao_julgador_nome_datajud,object


In [ ]:
stj_datajud_movimentos.dtypes


,0
numero_processo,object
numero_registro_stj,object
movimento_idx,int64
codigo_movimento,int64
nome_movimento,object
data_movimento,object
complementos_tabelados,object
orgao_julgador_codigo_movimento,object
orgao_julgador_nome_movimento,object


In [ ]:
stj_datajud_processos.head(3)
stj_datajud_movimentos.head(3)


,numero_processo,numero_registro_stj,movimento_idx,codigo_movimento,nome_movimento,data_movimento,complementos_tabelados,orgao_julgador_codigo_movimento,orgao_julgador_nome_movimento
0,00000094820228272722,202301327446,0,26,Distribuição,2023-04-27T16:15:08.000Z,"[{""codigo"": 2, ""valor"": 1, ""nome"": ""competênci...",None,None
1,00000094820228272722,202301327446,1,51,Conclusão,2023-04-27T16:30:05.000Z,"[{""codigo"": 3, ""valor"": 6, ""nome"": ""para decis...",None,None
2,00000094820228272722,202301327446,2,85,Petição,2023-05-23T19:31:00.000Z,"[{""codigo"": 19, ""valor"": 122, ""nome"": ""Parecer...",None,None


In [ ]:
stj_datajud_processos[[
    "numero_processo",
    "data_ajuizamento_datajud",
    "data_ultima_atualizacao_datajud"
]].head(10)


,numero_processo,data_ajuizamento_datajud,data_ultima_atualizacao_datajud
0,00000094820228272722,2023-04-26T00:00:00.000Z,2025-03-18T23:57:12.467Z
1,00000195520128020001,2023-06-24T00:00:00.000Z,2025-03-19T00:18:44.907Z
2,00000270520134047105,2017-03-16T00:00:00.000Z,2024-08-09T10:30:45.848Z
3,00000461120218120012,20230623000000,2026-02-04T20:07:25.666000Z
4,00000675420208190035,2023-02-09T00:00:00.000Z,2024-07-23T10:19:36.749Z
5,00000959119988050064,2023-06-23T00:00:00.000Z,2024-07-24T07:52:04.053Z
6,00001189320168160001,2023-06-16T00:00:00.000Z,2025-03-14T01:20:26.910Z
7,00001617220188180108,2023-06-26T00:00:00.000Z,2025-03-19T00:19:30.250Z
8,00001681020154036126,2021-10-28T00:00:00.000Z,2025-04-14T14:40:54.187Z
9,00002166520208050156,2023-05-01T00:00:00.000Z,2025-03-19T00:20:06.284Z


In [ ]:
stj_datajud_movimentos[[
    "numero_processo",
    "data_movimento"
]].head(10)


,numero_processo,data_movimento
0,00000094820228272722,2023-04-27T16:15:08.000Z
1,00000094820228272722,2023-04-27T16:30:05.000Z
2,00000094820228272722,2023-05-23T19:31:00.000Z
3,00000094820228272722,2023-06-07T16:56:00.000Z
4,00000094820228272722,2023-06-13T10:21:00.000Z
5,00000094820228272722,2023-06-13T10:30:32.000Z
6,00000094820228272722,2023-06-29T18:37:00.000Z
7,00000094820228272722,2023-06-30T16:30:16.000Z
8,00000094820228272722,2023-06-30T16:59:13.000Z
9,00000094820228272722,2023-07-03T09:21:00.000Z
